# Teste do TabPFN — imputação (TP2, Mineração de Dados)

Compara **TabPFN**, **MICE** e **mediana** numa amostra de 10 mil partidas, com os mesmos 5 folds e as mesmas máscaras (10% a 40% dos valores escondidos) do projeto `mineracao-futebol`.

**Antes de rodar:**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)**.
2. Tenha à mão o arquivo `amostra_10mil.csv` (pasta `notebooks/` do projeto).
3. Tenha a sua API key da Prior Labs: em https://ux.priorlabs.ai/account (a licença já foi aceita). O TabPFN pede essa chave na primeira vez.

Depois: **Ambiente de execução → Executar tudo**.

## 1. Instalação

In [ ]:
!pip install -q tabpfn

## 2. Enviar a amostra
Selecione o arquivo `amostra_10mil.csv`.

In [ ]:
from google.colab import files
enviado = files.upload()
assert "amostra_10mil.csv" in enviado, "Envie o arquivo amostra_10mil.csv"

## 3. Protocolo (o mesmo do projeto)

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer

# Mesmo protocolo de src/imputacao/protocolo.py do projeto
COLUNAS = ["HTHG", "HTAG", "HS", "AS", "HST", "AST", "HF", "AF",
           "HC", "AC", "HY", "AY", "HR", "AR"]
PERCENTUAIS = [0.10, 0.20, 0.30, 0.40]
KFOLD = KFold(n_splits=5, shuffle=True, random_state=42)


def gerar_mascara(fold, percentual, formato):
    """Máscara MCAR: mesma semente dos experimentos do projeto."""
    semente = 42 + fold * 100 + int(percentual * 100)
    return np.random.default_rng(semente).random(formato) < percentual


def calcular_metricas(metodo, percentual, fold, X_teste, X_imputado, mascara,
                      tempo_fit, tempo_transform):
    reais, imputados = np.asarray(X_teste), np.asarray(X_imputado)
    mse = mean_squared_error(reais[mascara], imputados[mascara])
    linhas = [{"metodo": metodo, "percentual": percentual * 100, "fold": fold,
               "variavel": "GERAL",
               "MAE": mean_absolute_error(reais[mascara], imputados[mascara]),
               "MSE": mse, "RMSE": np.sqrt(mse), "R2": np.nan,
               "tempo_fit": tempo_fit, "tempo_transform": tempo_transform,
               "tempo_total": tempo_fit + tempo_transform}]
    for i, coluna in enumerate(COLUNAS):
        m = mascara[:, i]
        real, imp = reais[m, i], imputados[m, i]
        mse_col = mean_squared_error(real, imp)
        linhas.append({"metodo": metodo, "percentual": percentual * 100,
                       "fold": fold, "variavel": coluna,
                       "MAE": mean_absolute_error(real, imp), "MSE": mse_col,
                       "RMSE": np.sqrt(mse_col), "R2": r2_score(real, imp),
                       "tempo_fit": np.nan, "tempo_transform": np.nan,
                       "tempo_total": np.nan})
    return linhas


X = pd.read_csv("amostra_10mil.csv")[COLUNAS]
assert X.shape == (10000, 14) and X.isna().sum().sum() == 0
print("Amostra:", X.shape)

## 4. TabPFN
Na primeira execução, o TabPFN baixa o modelo e pode pedir a API key: cole a chave de https://ux.priorlabs.ai/account na caixa que aparecer e aperte Enter.

In [ ]:
import torch
from tabpfn import TabPFNRegressor

N_ESTIMADORES = 4   # tamanho do ensemble do TabPFN
LOTE = 2000         # linhas previstas por chamada (reduza se faltar memória)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO, torch.cuda.get_device_name(0) if DISPOSITIVO == "cuda" else "")
assert DISPOSITIVO == "cuda", "Ative a GPU: Ambiente de execução > Alterar o tipo > GPU T4"


def imputar_tabpfn(X_treino, ausentes):
    """Um modelo por variável (alvo = a variável, entradas = as outras 13),
    com o fold de treino como contexto; prevê só as células escondidas."""
    imputados = {p: a.copy() for p, a in ausentes.items()}
    tempo_fit, tempo_transform = 0.0, {p: 0.0 for p in ausentes}
    for i, coluna in enumerate(COLUNAS):
        inicio = time.perf_counter()
        modelo = TabPFNRegressor(n_estimators=N_ESTIMADORES, device=DISPOSITIVO,
                                 random_state=42)
        modelo.fit(np.delete(X_treino, i, axis=1), X_treino[:, i])
        tempo_fit += time.perf_counter() - inicio
        for p, a in ausentes.items():
            linhas = np.where(np.isnan(a[:, i]))[0]
            if len(linhas) == 0:
                continue
            inicio = time.perf_counter()
            entradas = np.delete(a[linhas], i, axis=1)
            imputados[p][linhas, i] = np.concatenate(
                [modelo.predict(entradas[k:k + LOTE]) for k in range(0, len(entradas), LOTE)])
            tempo_transform[p] += time.perf_counter() - inicio
        del modelo
        torch.cuda.empty_cache()
    return imputados, tempo_fit, tempo_transform

## 5. Rodar a comparação (5 folds)

In [ ]:
resultados = []
inicio_geral = time.perf_counter()

for fold, (idx_treino, idx_teste) in enumerate(KFOLD.split(X), start=1):
    X_treino = X.iloc[idx_treino].to_numpy(dtype=float)
    X_teste = X.iloc[idx_teste].to_numpy(dtype=float)
    mascaras = {p: gerar_mascara(fold, p, X_teste.shape) for p in PERCENTUAIS}
    ausentes = {p: np.where(m, np.nan, X_teste) for p, m in mascaras.items()}

    for nome, imputador in [
        ("MICE", IterativeImputer(random_state=17, max_iter=5, tol=1e-2,
                                  initial_strategy="mean", imputation_order="random")),
        ("Mediana", SimpleImputer(strategy="median")),
    ]:
        inicio = time.perf_counter()
        imputador.fit(X_treino)
        tempo_fit = time.perf_counter() - inicio
        for p in PERCENTUAIS:
            inicio = time.perf_counter()
            X_imp = imputador.transform(ausentes[p])
            resultados += calcular_metricas(nome, p, fold, X_teste, X_imp, mascaras[p],
                                            tempo_fit, time.perf_counter() - inicio)

    imputados, tempo_fit, tempo_transform = imputar_tabpfn(X_treino, ausentes)
    for p in PERCENTUAIS:
        resultados += calcular_metricas("TabPFN", p, fold, X_teste, imputados[p],
                                        mascaras[p], tempo_fit, tempo_transform[p])

    print(f"Fold {fold} ok | {(time.perf_counter() - inicio_geral) / 60:.1f} min")

df = pd.DataFrame(resultados)
df.to_csv("teste_tabpfn.csv", index=False)
print("Resultados salvos em teste_tabpfn.csv")

## 6. Resultados

In [ ]:
from scipy.stats import wilcoxon

gerais = df[df["variavel"] == "GERAL"]
print("MAE médio (5 folds) — menor é melhor")
display(gerais.pivot_table(index="percentual", columns="metodo", values="MAE", aggfunc="mean").round(3))
print("Desvio do MAE entre folds")
display(gerais.pivot_table(index="percentual", columns="metodo", values="MAE", aggfunc="std").round(3))
print("Tempo médio por fold (s)")
display(gerais.pivot_table(index="percentual", columns="metodo", values="tempo_total", aggfunc="mean").round(1))
print("R² médio por variável (10% escondidos)")
display(df[(df["variavel"] != "GERAL") & (df["percentual"] == 10)]
        .pivot_table(index="variavel", columns="metodo", values="R2", aggfunc="mean").round(3))

# TabPFN x MICE: em quantos folds o TabPFN errou menos?
for p in [10, 20, 30, 40]:
    t = gerais[(gerais["percentual"] == p) & (gerais["metodo"] == "TabPFN")].sort_values("fold")["MAE"].to_numpy()
    m = gerais[(gerais["percentual"] == p) & (gerais["metodo"] == "MICE")].sort_values("fold")["MAE"].to_numpy()
    print(f"{p:.0f}%: TabPFN melhor que MICE em {(t < m).sum()}/5 folds | "
          f"diferença média {(t - m).mean():+.3f} | Wilcoxon p = {wilcoxon(t, m).pvalue:.4f}")

## 7. Baixar o resultado
Coloque o arquivo em `results/imputacao/teste_tabpfn.csv` no projeto.

In [ ]:
files.download("teste_tabpfn.csv")